In [ ]:
# Olist E-Commerce Data Engineering & Analytics

## Project Overview

This notebook demonstrates an end-to-end data preparation and ingestion pipeline using the Olist E-Commerce dataset.

### What is covered in this notebook:

- Loaded and inspected the Olist datasets using **Python and Pandas**.
- Checked the shape, duplicate records, missing values, and data types of the datasets.
- Cleaned missing values in the **Products** and **Reviews** datasets.
- Converted relevant columns to appropriate **datetime and integer data types**.
- Processed the large **Geolocation** dataset to remove redundancy and reduce its size.
- Aggregated geolocation data by `zip_code_prefix` using average latitude and longitude.
- Reduced the geolocation dataset from **1,000,163 rows to 19,015 rows (~98% reduction)**.
- Established a connection between Python and **SQL Server** using **PyODBC and SQLAlchemy**.
- Prepared the cleaned datasets as staging tables.
- Bulk loaded all **9 cleaned and transformed tables** into the `Olist_DB` SQL Server database.

### Tools & Technologies

**Python | Pandas | PyODBC | SQLAlchemy | SQL Server**

### Data Pipeline

**Raw CSV Files → Data Quality Checks → Data Cleaning → Data Transformation → Geolocation Optimization → SQL Server**

---

In [1]:

import os
import pandas as pd

pd.set_option('display.max_columns', None)   # show all columns when printing

In [4]:
# Folder that contains the 9 CSV files
base_path = "C:/Users/91974/OneDrive/Desktop/Olist E-Commerce Data Engineering & Analytics/Dataset/"
print(os.listdir(base_path))

['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


In [13]:
customers    = pd.read_csv(f'{base_path}olist_customers_dataset.csv')
orders       = pd.read_csv(f'{base_path}olist_orders_dataset.csv')
order_items  = pd.read_csv(f'{base_path}olist_order_items_dataset.csv')
payments     = pd.read_csv(f'{base_path}olist_order_payments_dataset.csv')
products     = pd.read_csv(f'{base_path}olist_products_dataset.csv')
translations = pd.read_csv(f'{base_path}product_category_name_translation.csv')
reviews      = pd.read_csv(f'{base_path}olist_order_reviews_dataset.csv')
sellers      = pd.read_csv(f'{base_path}olist_sellers_dataset.csv')

# Put them in a dictionary so we can loop over them
datasets = {
    'Customers': customers,
    'Orders': orders,
    'Order Items': order_items,
    'Payments': payments,
    'Products': products,
    'Translations': translations,
    'Reviews': reviews,
    'Sellers': sellers
}

# Shape check: (rows, columns)
for name, df in datasets.items():
    print(f"{name:<14} {df.shape}")

Customers      (99441, 5)
Orders         (99441, 8)
Order Items    (112650, 7)
Payments       (103886, 5)
Products       (32951, 9)
Translations   (71, 2)
Reviews        (99224, 7)
Sellers        (3095, 4)


In [14]:
# Check for duplicate rows in each dataset
for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    print(f"{name} Table has {duplicate_count} duplicate rows.")
    
    # If duplicates exist, drop them in-place
    if duplicate_count > 0:
        df.drop_duplicates(inplace=True)
        print(f"--> Fixed: Removed duplicates from {name} Table.")

Customers Table has 0 duplicate rows.
Orders Table has 0 duplicate rows.
Order Items Table has 0 duplicate rows.
Payments Table has 0 duplicate rows.
Products Table has 0 duplicate rows.
Translations Table has 0 duplicate rows.
Reviews Table has 0 duplicate rows.
Sellers Table has 0 duplicate rows.


In [15]:
# Loop through the dictionary and print missing values for each DataFrame
for name, df in datasets.items():
    null_counts = df.isnull().sum()
    # Filter to show only columns that have at least 1 null value
    missing_columns = null_counts[null_counts > 0]
    
    print(f"==================== {name.upper()} MISSING VALUES ====================")
    if not missing_columns.empty:
        print(missing_columns)
    else:
        print("No missing values found in this table.")
    print("\n" + "_"*60 + "\n")

==================== CUSTOMERS MISSING VALUES ====================
No missing values found in this table.

____________________________________________________________

==================== ORDERS MISSING VALUES ====================
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

____________________________________________________________

==================== ORDER ITEMS MISSING VALUES ====================
No missing values found in this table.

____________________________________________________________

==================== PAYMENTS MISSING VALUES ====================
No missing values found in this table.

____________________________________________________________

==================== PRODUCTS MISSING VALUES ====================
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
pr

In [16]:
# Fill categorical column with 'unknown'
products['product_category_name'] = products['product_category_name'].fillna('unknown')

# Fill numerical product specs with 0
num_product_cols = ['product_name_lenght', 'product_description_lenght', 'product_photos_qty', 
                    'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in num_product_cols:
    products[col] = products[col].fillna(0)

# Verify that there are absolutely no missing values left in the entire products DataFrame
print("Total missing values in Products table:", products.isnull().sum().sum())

Total missing values in Products table: 0


In [17]:
# Fill text columns with standard placeholder text
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('No Title')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('No Message')

# Verify that there are no missing values left in the review comment columns
print("Missing values in comment titles:", reviews['review_comment_title'].isnull().sum())
print("Missing values in comment messages:", reviews['review_comment_message'].isnull().sum())

Missing values in comment titles: 0
Missing values in comment messages: 0


In [18]:
# Check current data types for Orders table
print(orders.dtypes)

order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object


In [19]:
# List of columns that need to be converted to datetime
orders_date_cols = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

# Convert columns to datetime
for col in orders_date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# Verify the changes for this table specifically
print("\n--- Verified Data Types for Orders Table ---")
print(orders[orders_date_cols].dtypes)


--- Verified Data Types for Orders Table ---
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [20]:
# Print the data types of the orders table after conversion
print(orders.dtypes)

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [21]:
# Check current data types for Reviews table
print(reviews.dtypes)

review_id                  object
order_id                   object
review_score                int64
review_comment_title       object
review_comment_message     object
review_creation_date       object
review_answer_timestamp    object
dtype: object


In [22]:
# List of columns that need to be converted to datetime in Reviews
reviews_date_cols = ['review_creation_date', 'review_answer_timestamp']

# Convert columns to datetime with error handling
for col in reviews_date_cols:
    reviews[col] = pd.to_datetime(reviews[col], errors='coerce')

# Verify the changes for this table specifically
print("\n--- Verified Data Types for Reviews Table ---")
print(reviews[reviews_date_cols].dtypes)


--- Verified Data Types for Reviews Table ---
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object


In [23]:
# Check current data types for Reviews table
print(reviews.dtypes)

review_id                          object
order_id                           object
review_score                        int64
review_comment_title               object
review_comment_message             object
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object


In [24]:
# Check current data types for Products table
print(products.dtypes)

product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object


In [25]:
# List of columns to convert from float to int
cols_to_int = ['product_name_lenght', 'product_description_lenght', 'product_photos_qty']

# Convert using astype()
for col in cols_to_int:
    products[col] = products[col].astype('int64')

# Verify the changes
print("--- Verified Data Types for Products Table ---")
print(products[cols_to_int].dtypes)

--- Verified Data Types for Products Table ---
product_name_lenght           int64
product_description_lenght    int64
product_photos_qty            int64
dtype: object


In [26]:
# Check current data types for Products table
print(products.dtypes)

product_id                     object
product_category_name          object
product_name_lenght             int64
product_description_lenght      int64
product_photos_qty              int64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object


In [27]:
remaining_datasets = {
    'Customers': customers,
    'Order Items': order_items,
    'Payments': payments,
    'Sellers': sellers,
    'Translations': translations
}

for name, df in remaining_datasets.items():
    print(f"--- {name} Table Data Types ---")
    print(df.dtypes)
    print("\n" + "_"*40 + "\n")

--- Customers Table Data Types ---
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

________________________________________

--- Order Items Table Data Types ---
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

________________________________________

--- Payments Table Data Types ---
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

________________________________________

--- Sellers Table Data Types ---
seller_id                 object
seller_zip_code_prefix     int64
seller_city               object
seller_state              object
dtype:

In [31]:
# Convert shipping_limit_date to datetime in Order Items table
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'], errors='coerce')

# Verify the fix
print("Order Items Table - Shipping Limit Date Type:")
print(order_items['shipping_limit_date'].dtypes)

Order Items Table - Shipping Limit Date Type:
datetime64[ns]


In [32]:
print(order_items.dtypes)

order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object


In [ ]:
## For Geolocation Dataset
The `olist_geolocation_dataset.csv` is an extremely heavy file containing **over 1 Million rows** of spatial coordinates ($Latitude$ and $Longitude$). 
* Loading this dataset initially with the other tables would unnecessarily drain system memory (RAM).
* **Data Redundancy:** Upon deep inspection, the dataset contains massive duplication where the exact same `zip_code_prefix` is repeated dozens of times with micro-differences in coordinates (just a few meters apart in the same street).
* Uploading 1M+ rows to SQL Server and feeding them into Power BI would severely degrade query performance and slow down dashboard rendering, without adding any real business value.

In [ ]:
## Geolocation Data Compression
* **Problem:** The raw geolocation dataset is extremely heavy and contains redundant rows for the same zip codes.
* **Action:** 
  * Group data by `zip_code_prefix` and calculate the average latitude and longitude (`mean`) to compress the file size.
  * Rename columns to clean, standardized names (`zip_code_prefix`, `lat`, `lng`, `city`, `state`).
  * Upload the final compressed table directly to SQL Server as `stg_geolocation`.

In [33]:
# 1. Load the heavy geolocation file
geo = pd.read_csv(f'{base_path}olist_geolocation_dataset.csv')

# 2. Aggregate coordinates by zip_code_prefix to remove redundant rows
# We take the mean (average) lat and lng for each unique zip code
print("Aggregating coordinates to reduce file size...")
geo_cleaned = geo.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean',
    'geolocation_city': 'first',  # Keep the first city name associated
    'geolocation_state': 'first'  # Keep the first state name associated
}).reset_index()

# Rename columns to make them clean and standard
geo_cleaned.columns = ['zip_code_prefix', 'lat', 'lng', 'city', 'state']

print(f"Original rows: {len(geo)} | Cleaned/Compressed rows: {len(geo_cleaned)}")

Aggregating coordinates to reduce file size...
Original rows: 1000163 | Cleaned/Compressed rows: 19015


In [ ]:
> ### Compression Summary:
> * **Original Rows:** 1,000,163
> * **Cleaned/Compressed Rows:** 19,015
> * **Reduction Rate:** **~98%** decrease in data volume, ensuring faster queries and optimized storage in SQL Server.

In [35]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [36]:
import urllib
from sqlalchemy import create_engine

In [37]:
# 1. Connection Parameters
server = r'localhost\SQLEXPRESS'
database = 'Olist_DB'
driver = 'ODBC Driver 17 for SQL Server'

# 2. Create the SQLAlchemy Database Engine
params = urllib.parse.quote_plus(
    f"DRIVER={{{driver}}};SERVER={server};DATABASE={database};"
    f"Trusted_Connection=yes;TrustServerCertificate=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# 3. Test
with engine.connect() as conn:
    print("Connected to:", conn.connection.getinfo(pyodbc.SQL_DATABASE_NAME))

Connected to: Olist_DB


In [38]:
print("customers:   ", customers.shape)
print("orders:      ", orders.shape, "| purchase col:", orders['order_purchase_timestamp'].dtype)
print("order_items: ", order_items.shape)
print("payments:    ", payments.shape)
print("products:    ", products.shape)
print("translations:", translations.shape)
print("reviews:     ", reviews.shape)
print("sellers:     ", sellers.shape)
print("geo_cleaned: ", geo_cleaned.shape)

customers:    (99441, 5)
orders:       (99441, 8) | purchase col: datetime64[ns]
order_items:  (112650, 7)
payments:     (103886, 5)
products:     (32951, 9)
translations: (71, 2)
reviews:      (99224, 7)
sellers:      (3095, 4)
geo_cleaned:  (19015, 5)


In [39]:
# 3. Consolidate all 9 cleaned DataFrames into a pipeline dictionary
final_tables = {
    'stg_customers': customers,
    'stg_orders': orders,
    'stg_order_items': order_items,
    'stg_payments': payments,
    'stg_products': products,
    'stg_translations': translations,
    'stg_reviews': reviews,
    'stg_sellers': sellers,
    'stg_geolocation': geo_cleaned  # Optimized and compressed spatial dataset
}

# 4. Execute the Bulk Upload Pipeline
print("Starting the Final Bulk Upload Pipeline to SQL Server...\n")

for table_name, df in final_tables.items():
    print(f"Uploading {table_name} ({len(df)} rows)...")
    df.to_sql(name=table_name, con=engine, if_exists='replace', index=False, chunksize=5000)
    print(f"{table_name} uploaded successfully!\n")

print("=" * 50)
print("SUCCESS: All 9 tables are now live inside Olist_DB!")

Starting the Final Bulk Upload Pipeline to SQL Server...

Uploading stg_customers (99441 rows)...
stg_customers uploaded successfully!

Uploading stg_orders (99441 rows)...
stg_orders uploaded successfully!

Uploading stg_order_items (112650 rows)...
stg_order_items uploaded successfully!

Uploading stg_payments (103886 rows)...
stg_payments uploaded successfully!

Uploading stg_products (32951 rows)...
stg_products uploaded successfully!

Uploading stg_translations (71 rows)...
stg_translations uploaded successfully!

Uploading stg_reviews (99224 rows)...
stg_reviews uploaded successfully!

Uploading stg_sellers (3095 rows)...
stg_sellers uploaded successfully!

Uploading stg_geolocation (19015 rows)...
stg_geolocation uploaded successfully!

SUCCESS: All 9 tables are now live inside Olist_DB!
